In [6]:

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import time
from glob import glob
from dask import compute as dask_compute

TFB_CSV = "/g/data/gb02/sm5259/TFB_Objects/TFB_days.csv"
FIG_DIR = "/g/data/gb02/sm5259/TFB_Objects/analysis/figures"
COMPOSITE_DIR = "/g/data/gb02/sm5259/TFB_Objects/analysis/composites"
WEATHER_OBJECT_DIR = "/g/data/if69/cj0591/GC26_energy_synoptics/data/weatherfeatures.era5"

CLIM_START = "1986-01-01"
CLIM_END = "2022-12-31"
FIRE_SEASON_MONTHS = {10, 11, 12, 1, 2, 3, 4}
ANOMALY_VMAX = 0.15

LON_BOUNDS = slice(140, 154)
LAT_BOUNDS = slice(-39, -28)
MAP_EXTENT = [140, 154, -39, -28]

LAT_VALUES = np.arange(-90, 90.5, 0.5)

SINGLE_OBJECTS = {
    "Cyclone":     {"path": f"{WEATHER_OBJECT_DIR}/mincl/cdf/*/C*.nc",         "var": "INPUT"},
    "Anticyclone": {"path": f"{WEATHER_OBJECT_DIR}/maxcl/cdf/*/A*.nc",         "var": "FLAG"},
    "Front 700":   {"path": f"{WEATHER_OBJECT_DIR}/fronts/cdf.700hPa/*/F*.nc", "var": "FRONT"},
    "Front 850":   {"path": f"{WEATHER_OBJECT_DIR}/fronts/cdf.850hPa/*/F*.nc", "var": "FRONT"},
}

WCB_CONFIG = {
    "path": f"{WEATHER_OBJECT_DIR}/wcb/cdf.1hourly/*/hit_*.nc",
    "variables": {
        "WCB inflow":  "GT800",
        "WCB ascent":  "MIDTROP",
        "WCB outflow": "LT400",
    },
}


def select_fire_season_files(path_pattern):
    #Return only files for fire-season months (Oct-Apr) from a glob pattern.
    all_files = sorted(glob(path_pattern))
    selected = []
    for filepath in all_files:
        filename = os.path.basename(filepath)
        month_str = filename.replace(".nc", "").split("_")[-1]
        try:
            if int(month_str) in FIRE_SEASON_MONTHS:
                selected.append(filepath)
        except ValueError:
            selected.append(filepath)
    return selected


def assign_weatherfeature_coords(raw_ds):
    #Attach real lat/lon to weather object datasets (from 21CW group notebook).
    raw_ds = raw_ds.squeeze()
    y_dim_name = list(raw_ds.dims)[1]
    x_dim_name = list(raw_ds.dims)[2]
    n_lon = raw_ds.sizes[x_dim_name]

    lon_values = -180 + 0.5 * np.arange(n_lon)

    raw_ds = raw_ds.assign_coords({y_dim_name: (y_dim_name, LAT_VALUES)})
    raw_ds = raw_ds.assign_coords({x_dim_name: (x_dim_name, lon_values)})
    raw_ds = raw_ds.rename({y_dim_name: "latitude", x_dim_name: "longitude"})

    if raw_ds.longitude.size != 720:
        raw_ds = raw_ds.sel(longitude=np.arange(-180, 180, 0.5))

    return raw_ds


def compute_composites(daily_presence, tfb_dates_in_clim_period):
    #Compute climatology, TFB composite, and anomaly in a single graph execution.
    climatology_field = daily_presence.mean(dim="time")
    tfb_day_presence = daily_presence.sel(time=tfb_dates_in_clim_period)
    tfb_composite_field = tfb_day_presence.mean(dim="time")
    anomaly_field = tfb_composite_field - climatology_field

    climatology_field, tfb_composite_field, anomaly_field = dask_compute(
        climatology_field, tfb_composite_field, anomaly_field
    )
    return climatology_field, tfb_composite_field, anomaly_field


def save_composite(climatology_field, tfb_composite_field, anomaly_field,
                   object_name, n_tfb_days):
    #Save the three composite fields to a compressed NetCDF file.
    safe_name = object_name.lower().replace(" ", "_")
    composite_ds = xr.Dataset({
        "climatology": climatology_field,
        "tfb_composite": tfb_composite_field,
        "anomaly": anomaly_field,
    })
    composite_ds.attrs = {
        "object": object_name,
        "clim_start": CLIM_START,
        "clim_end": CLIM_END,
        "n_tfb_days": n_tfb_days,
        "fire_season_months": "Oct-Apr",
    }
    encoding = {
        var: {"zlib": True, "complevel": 4, "dtype": "float32"}
        for var in composite_ds.data_vars
    }
    nc_path = f"{COMPOSITE_DIR}/composite_{safe_name}.nc"
    composite_ds.to_netcdf(nc_path, encoding=encoding)
    print(f"  Saved composite: {nc_path}")


def plot_three_panel_map(climatology_field, tfb_composite_field, anomaly_field,
                         object_name, n_tfb_days):
    #Three-panel map: climatology | TFB composite | anomaly.
    fig, axes = plt.subplots(
        1, 3, figsize=(18, 5),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )

    panels = [
        (climatology_field,   f"Climatology\n(Oct-Apr, 1986-2022)", "YlOrRd"),
        (tfb_composite_field, f"TFB composite (n={n_tfb_days})",    "YlOrRd"),
        (anomaly_field,       "Anomaly (TFB - climatology)",        "RdBu_r"),
    ]

    for ax, (field, title, cmap) in zip(axes, panels):
        ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.5, linestyle="--")

        plot_kwargs = {"cmap": cmap, "transform": ccrs.PlateCarree()}
        if "Anomaly" in title:
            plot_kwargs["vmin"] = -ANOMALY_VMAX
            plot_kwargs["vmax"] = ANOMALY_VMAX
        else:
            plot_kwargs["vmin"] = 0

        mesh = ax.pcolormesh(
            field.longitude, field.latitude, field.values, **plot_kwargs
        )
        ax.set_title(title, fontsize=11)
        plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.06,
                     label="Daily occurrence frequency")

    fig.suptitle(f"Weather Object Composite: {object_name}", fontsize=14, y=1.02)
    fig.tight_layout()

    safe_name = object_name.lower().replace(" ", "_")
    output_path = f"{FIG_DIR}/composite_anomaly_{safe_name}.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved figure: {output_path}")


def process_object(object_name, daily_presence, tfb_dates_in_clim_period, n_tfb_days):
    #Compute composites, save NetCDF, and plot for one object.
    print("  Computing composites...")
    climatology_field, tfb_composite_field, anomaly_field = compute_composites(
        daily_presence, tfb_dates_in_clim_period
    )
    save_composite(climatology_field, tfb_composite_field, anomaly_field,
                   object_name, n_tfb_days)
    plot_three_panel_map(climatology_field, tfb_composite_field, anomaly_field,
                         object_name, n_tfb_days)


def log_progress(object_start_time, elapsed_times, objects_done, n_total):
    #Print elapsed time and estimated remaining time.
    elapsed = time.time() - object_start_time
    elapsed_times.append(elapsed)
    est_remaining = np.mean(elapsed_times) * (n_total - objects_done)
    print(f"  Took {elapsed/60:.1f} min | ~{est_remaining/60:.0f} min remaining")


def main():
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(COMPOSITE_DIR, exist_ok=True)
    total_start_time = time.time()

    tfb_df = pd.read_csv(TFB_CSV, parse_dates=["Date"])
    all_tfb_dates = pd.DatetimeIndex(tfb_df["Date"])
    tfb_dates_in_clim_period = all_tfb_dates[
        (all_tfb_dates >= CLIM_START) & (all_tfb_dates <= CLIM_END)
    ]
    n_tfb_days = len(tfb_dates_in_clim_period)
    print(f"TFB dates in climatology period: {n_tfb_days} of {len(all_tfb_dates)} total")

    # Pre-build all file lists to avoid repeated filesystem scans
    file_lists = {
        name: select_fire_season_files(info["path"])
        for name, info in SINGLE_OBJECTS.items()
    }
    file_lists["wcb"] = select_fire_season_files(WCB_CONFIG["path"])

    n_total = len(SINGLE_OBJECTS) + len(WCB_CONFIG["variables"])
    objects_done = 0
    elapsed_times = []

    for object_name, object_info in SINGLE_OBJECTS.items():
        object_start_time = time.time()
        objects_done += 1
        print(f"\n[{objects_done}/{n_total}] {object_name}")

        print(f"  Opening {len(file_lists[object_name])} fire-season files...")
        raw_ds = xr.open_mfdataset(
            file_lists[object_name], combine="by_coords",
            chunks={"time": 24 * 90}, parallel=True,
        )
        raw_ds = assign_weatherfeature_coords(raw_ds)
        raw_ds = raw_ds.sel(latitude=LAT_BOUNDS, longitude=LON_BOUNDS)
        raw_ds = raw_ds.sel(time=slice(CLIM_START, CLIM_END))

        print("  Computing daily presence...")
        daily_presence = (raw_ds[object_info["var"]] != 0).resample(time="1D").any()

        process_object(object_name, daily_presence, tfb_dates_in_clim_period, n_tfb_days)
        log_progress(object_start_time, elapsed_times, objects_done, n_total)

    # WCB: open once, resample all three variables together, then process each
    print(f"\nOpening WCB files (shared by inflow/ascent/outflow)...")
    wcb_vars = list(WCB_CONFIG["variables"].values())
    print(f"  Opening {len(file_lists['wcb'])} fire-season files...")
    wcb_raw_ds = xr.open_mfdataset(
        file_lists["wcb"], combine="by_coords",
        chunks={"time": 24 * 90}, parallel=True,
    )
    wcb_raw_ds = assign_weatherfeature_coords(wcb_raw_ds)
    wcb_raw_ds = wcb_raw_ds.sel(latitude=LAT_BOUNDS, longitude=LON_BOUNDS)
    wcb_raw_ds = wcb_raw_ds.sel(time=slice(CLIM_START, CLIM_END))

    print("  Resampling all WCB variables to daily (once)...")
    wcb_daily = (wcb_raw_ds[wcb_vars] != 0).resample(time="1D").any()

    for object_name, var_name in WCB_CONFIG["variables"].items():
        object_start_time = time.time()
        objects_done += 1
        print(f"\n[{objects_done}/{n_total}] {object_name}")

        process_object(object_name, wcb_daily[var_name],
                       tfb_dates_in_clim_period, n_tfb_days)
        log_progress(object_start_time, elapsed_times, objects_done, n_total)

    print(f"\nDone. Total time: {(time.time() - total_start_time)/60:.1f} min")


if __name__ == "__main__":
    main()

TFB dates in climatology period: 198 of 215 total

[1/7] Cyclone
  Opening 509 fire-season files...


KeyboardInterrupt: 

In [7]:
# Resume: WCB ascent and outflow only
wcb_fire_season_files = select_fire_season_files(WCB_CONFIG["path"])
print(f"Opening {len(wcb_fire_season_files)} WCB files...")
wcb_raw_ds = xr.open_mfdataset(
    wcb_fire_season_files, combine="by_coords",
    chunks={"time": 24 * 90}, parallel=True,
)
wcb_raw_ds = assign_weatherfeature_coords(wcb_raw_ds)
wcb_raw_ds = wcb_raw_ds.sel(latitude=LAT_BOUNDS, longitude=LON_BOUNDS)
wcb_raw_ds = wcb_raw_ds.sel(time=slice(CLIM_START, CLIM_END))

print("Resampling to daily...")
wcb_daily = (wcb_raw_ds[["MIDTROP", "LT400"]] != 0).resample(time="1D").any().compute()

tfb_df = pd.read_csv(TFB_CSV, parse_dates=["Date"])
all_tfb_dates = pd.DatetimeIndex(tfb_df["Date"])
tfb_dates_in_clim_period = all_tfb_dates[
    (all_tfb_dates >= CLIM_START) & (all_tfb_dates <= CLIM_END)
]
n_tfb_days = len(tfb_dates_in_clim_period)

for object_name, var_name in [("WCB ascent", "MIDTROP"), ("WCB outflow", "LT400")]:
    print(f"\n{object_name}")
    process_object(object_name, wcb_daily[var_name], tfb_dates_in_clim_period, n_tfb_days)

print("Done.")

Opening 301 WCB files...
Resampling to daily...

WCB ascent
  Computing composites...
  Saved composite: /g/data/gb02/sm5259/TFB_Objects/analysis/composites/composite_wcb_ascent.nc
  Saved figure: /g/data/gb02/sm5259/TFB_Objects/analysis/figures/composite_anomaly_wcb_ascent.png

WCB outflow
  Computing composites...
  Saved composite: /g/data/gb02/sm5259/TFB_Objects/analysis/composites/composite_wcb_outflow.nc
  Saved figure: /g/data/gb02/sm5259/TFB_Objects/analysis/figures/composite_anomaly_wcb_outflow.png
Done.
